# Motivation and introduction

This file provides examples on how to use config system.

The config system provides a unified way to configure parameters for scripts that use the library.  
It supports reading from files, the commandline, environment vars, overrides in the code itself and also editors.  
for details on the implementation, requirements and rejected alternatives, see [devnotes](../900_dev/feature_cfg.md).

In [1]:
# imports and preparations
from pylib.lib.fns import getcfg
# lets enable autoreload so we can use these scrips for interactive src-debugging.
%load_ext autoreload
%autoreload 2

In [2]:
# At this point, the cfg is empty and no sources are registered

cfg = getcfg()

cfg.values.print()
cfg.sources.print()

cfg

   Config sources   
┏━━━┳━━━━━━━┳━━━━━━┓
┃ # ┃ title ┃ prio ┃
┡━━━╇━━━━━━━╇━━━━━━┩
└───┴───────┴──────┘

# writing values via files

## json

The easiest way to write values is via json. This is equivalent to passing dicts directly.  
As with every type of input, all values that dont exist before reading get created. On unmounting a file, they dont get destroyed.
#TODO: is this the behavior we want, or do we want them destroyed on unmount?

In [3]:
#create some dicts.
dct1 = {"val1":1,"val2": "test", "val3": ["this","is","a","list"]}
dct2 = {"testkey": "testvalue", "val1": 2}
dct3 = {"val1":3, "nested": "test","val4": "bla"}

In [4]:
#setting values
#remember: passing these dicts is the same as passing a path to a json file with the same content
cfg.clear() # use this to clear the cfg, we use it here so we can run the cell repeatedly
cfg.set(dct1)

cfg.values.print()
cfg.sources.print()

cfg
├── val1    1
├── val2    test
└── val3    ['this', 'is', 'a', 'list']

         Config sources          
┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ # ┃ title         ┃      prio ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ 0 │ cfg_825c02084 │ (1, 4, 0) │
└───┴───────────────┴───────────┘

In [5]:
# note: you can also get the raw cfg xml:
print(cfg.toxml(dst=str)) # dst=str returns string, otherwise goes to xml file

cfg.set(dct2)

#note how "val1" got overridden
cfg.get("val1")
cfg.values.print()
cfg.sources.print()

<cfg>
	<val1 cfg_parent="" cfg_type="int" cfg_source="0">1</val1>
	<val2 cfg_parent="" cfg_type="str" cfg_source="0">test</val2>
	<val3 cfg_parent="" cfg_type="list" cfg_source="0">['this', 'is', 'a', 'list']</val3>
</cfg>


cfg
├── val1    1
├── val2    test
├── val3    ['this', 'is', 'a', 'list']
├── testkey testvalue
└── val1    2

          Config sources          
┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ # ┃ title         ┃       prio ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ 0 │ cfg_825c02084 │  (1, 4, 0) │
│ 1 │ cfg_37985a218 │ (0, 4, -1) │
└───┴───────────────┴────────────┘

In [6]:
cfg.sources.unmount(1) #remove source by index (also possible by name)

#the override is removed, but the created values still exist.
cfg.values.print()
cfg.sources.print()
print(cfg.get("val1"))

cfg
├── val1    1
├── val2    test
├── val3    ['this', 'is', 'a', 'list']
├── testkey testvalue
└── val1    2

         Config sources          
┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ # ┃ title         ┃      prio ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ 0 │ cfg_825c02084 │ (1, 4, 0) │
└───┴───────────────┴───────────┘

{'cfg/val1[1]': '1', 'cfg/val1[2]': '2'}


In [7]:
#nested values
#lets load a dict into a subnode
cfg.set(dct3,parent="nest") #parent gets created if it doesnt exist

cfg.values.print()
print(cfg.get("nest/val1"))

cfg
├── val1    1
├── val2    test
├── val3    ['this', 'is', 'a', 'list']
└── nest
    └── val1    3

3


# reading values (more options than discussed above)

In [8]:
# defaults
print(cfg.get("doesnt_exist", default = 1))
print(cfg.get("also_doesnt_exist"))

1
None


In [9]:
# simple queries
print(cfg.get("nest/*"))
print(cfg.get("nest/val*"))

3


XPathEvalError: Invalid expression